Derived Features

| New Feature | Formula | Rationale |
| :--- | :--- | :--- |
| **avg_recharge_amount** | montant / frequence_rech | Value per recharge event |
| **avg_revenue_per_tx** | revenue / frequence | Revenue efficiency |
| **is_data_user** | (data_volume > 0).astype(int) | Binary: engaged with data |
| **no_data_flag** | data_volume.isna().astype(int) | Missing = never used data |
| **network_quality_delta** | region_nqs - arr_nqs | Local vs sub-region quality gap |
| **log_data_volume** | np.log1p(data_volume) | Tame extreme skew |
| **log_on_net** | np.log1p(on_net) | Tame extreme skew |
| **log_revenue** | np.log1p(revenue) | Normalize for linear models |

***
Drop / Flag Recommendations

| Column | Action | Reason | Omar_Notes|
| :--- | :--- | :--- | :--- |
| **user_id** | DROP | Identifier, no signal | became the `index` of the dataset.|
| **mrg** | DROP | Zero-variance |
| **zone1, zone2** | DROP | >92% missing | _0_0_|
| **region_avg_signal** | DROP | Zero-variance (all 0) |
| **region_signal_strength_index** | DROP | Zero-variance (all 0) |
| **dept_signal_strength_index** | DROP | Zero-variance (all 0) |
| **arr_signal_strength_index** | DROP | Zero-variance (all 0) |
| **region/dept/arr_coverage_idx** | KEEP ONE | Identical values — select region_coverage_index |
| **tenure** | RECODE | Near-constant; binarize to loyal/new | decomposed to symbol and min_value of corresponding range. |
| **top_pack** | ENCODE | Target-encode (90 levels too many for OHE) | kept the original beside its codec.|

In [2]:
import pandas as pd
import numpy as np

tele_set = pd.read_csv('https://drive.google.com/uc?id=14Z9IFFAySwBSCKtAkhPUzR3-HwAoj_7a').drop('Unnamed: 0',axis = 1)#NOTE: id was added by default
tele_set = tele_set.set_index('user_id',drop = True)
tele_set.info()

<class 'pandas.core.frame.DataFrame'>
Index: 91430 entries, 22322777e909ae0aa1c69e7b1fce40f333187fa6 to 32e30b9357c926370bb851cb2136f322aa2a574e
Data columns (total 31 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   region_CLEANED                            91430 non-null  object 
 1   tenure                                    91430 non-null  object 
 2   montant_CLEANED                           91430 non-null  int64  
 3   frequence_rech_CLEANED                    91430 non-null  int64  
 4   revenue_CLEANED                           91430 non-null  int64  
 5   arpu_segment_CLEANED                      91430 non-null  int64  
 6   frequence_CLEANED                         91430 non-null  int64  
 7   data_volume_CLEANED                       91430 non-null  int64  
 8   on_net_CLEANED                            91430 non-null  int64  
 9   orange_CLEANED                      

In [3]:
#Drop 0-valued columns
almost_cleaned_teleset = tele_set.drop([col for col in tele_set.columns if not tele_set[col].sum()],axis = 1)
#Merge symmetrical columns into one:
# coverage_index = almost_cleaned_teleset['region_coverage_index_CLEANED']
almost_cleaned_teleset = almost_cleaned_teleset.rename(columns = {'region_coverage_index_CLEANED': 'cvrg_index_CLEANED'})
almost_cleaned_teleset = almost_cleaned_teleset.drop([col for col in almost_cleaned_teleset.columns if 'coverage' in col],axis = 1)

print(almost_cleaned_teleset.info())
# almost_cleaned_teleset = almost_cleaned_teleset.T.drop_duplicates().T NOOOOOOOOOTE: although it is general, but destructive to the schema metadata (i.e. data types for the AI model )
#TBC... next cell: -

<class 'pandas.core.frame.DataFrame'>
Index: 91430 entries, 22322777e909ae0aa1c69e7b1fce40f333187fa6 to 32e30b9357c926370bb851cb2136f322aa2a574e
Data columns (total 25 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   region_CLEANED                            91430 non-null  object 
 1   tenure                                    91430 non-null  object 
 2   montant_CLEANED                           91430 non-null  int64  
 3   frequence_rech_CLEANED                    91430 non-null  int64  
 4   revenue_CLEANED                           91430 non-null  int64  
 5   arpu_segment_CLEANED                      91430 non-null  int64  
 6   frequence_CLEANED                         91430 non-null  int64  
 7   data_volume_CLEANED                       91430 non-null  int64  
 8   on_net_CLEANED                            91430 non-null  int64  
 9   orange_CLEANED                      

In [ ]:
#Transforming the CODEC of `top_pack` and `tenure`, in the SAME position
""" Solution from Gemini
# 1. Capture the original position of the attribute
col_idx = df.columns.get_loc("full_name")

# 2. Decompose the attribute (expand=True creates a DataFrame)
split_data = df["full_name"].str.split(" ", n=1, expand=True)

# 3. Drop the original column to make room
df.drop(columns="full_name", inplace=True)

# 4. Insert the new columns back at the original position
# We insert in reverse order or increment the index so they stay together
df.insert(col_idx, "first_name", split_data[0])
df.insert(col_idx + 1, "last_name", split_data[1])

Type: Categorical (String)
Description:
Represents the duration of the customer's relationship with Expresso.
Values are grouped into tenure ranges such as:

"""
tenure_idx = almost_cleaned_teleset.columns.get_loc('tenure')
duration = []
du_code = []
for value in almost_cleaned_teleset['tenure']: #TODO: more advanced, make it a set and then fill the whole range by reversed indexing of the original column ;)
  if '>' in value:
    duration.append(25)
    du_code.append(value.split()[0])
  else:
    others = value.split()
    du_code.append(others[0])

    if du_code[-1] == 'D':
      duration.append(6)
    elif du_code[-1] == 'E':
      duration.append(9)
    elif du_code[-1] == 'F':
      duration.append(12)
    elif du_code[-1] == 'G':
      duration.append(15)
    elif du_code[-1] == 'H':
      duration.append(18)
    elif du_code[-1] == 'I':
      duration.append(21)
    elif du_code[-1] == 'J':
      duration.append(24)

almost_cleaned_teleset = almost_cleaned_teleset.drop('tenure', axis = 1)

almost_cleaned_teleset.insert(tenure_idx,"tenure_symbol", du_code)
almost_cleaned_teleset.insert(tenure_idx + 1,"tenure_value", duration)

    #else: less_than 3 months, but not encountered actually*

In [5]:
almost_cleaned_teleset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 91430 entries, 22322777e909ae0aa1c69e7b1fce40f333187fa6 to 32e30b9357c926370bb851cb2136f322aa2a574e
Data columns (total 26 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   region_CLEANED                            91430 non-null  object 
 1   tenure_symbol                             91430 non-null  object 
 2   tenure_value                              91430 non-null  int64  
 3   montant_CLEANED                           91430 non-null  int64  
 4   frequence_rech_CLEANED                    91430 non-null  int64  
 5   revenue_CLEANED                           91430 non-null  int64  
 6   arpu_segment_CLEANED                      91430 non-null  int64  
 7   frequence_CLEANED                         91430 non-null  int64  
 8   data_volume_CLEANED                       91430 non-null  int64  
 9   on_net_CLEANED                      

In [7]:
[list(almost_cleaned_teleset['top_pack_CLEANED']).count(val) for val in set(almost_cleaned_teleset['top_pack_CLEANED'])]

from sklearn.preprocessing import LabelEncoder
# Apply Label Encoding
encoder = LabelEncoder()
almost_cleaned_teleset.insert(almost_cleaned_teleset.columns.get_loc('top_pack_CLEANED') + 1,"TOP_PACK_id",
                              encoder.fit_transform(almost_cleaned_teleset['top_pack_CLEANED'])
                              )

In [8]:
fully_cleaned_teleset = almost_cleaned_teleset.drop(columns = ['zone1_CLEANED','zone2_CLEANED'])

In [9]:
fully_cleaned_teleset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 91430 entries, 22322777e909ae0aa1c69e7b1fce40f333187fa6 to 32e30b9357c926370bb851cb2136f322aa2a574e
Data columns (total 25 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   region_CLEANED                            91430 non-null  object 
 1   tenure_symbol                             91430 non-null  object 
 2   tenure_value                              91430 non-null  int64  
 3   montant_CLEANED                           91430 non-null  int64  
 4   frequence_rech_CLEANED                    91430 non-null  int64  
 5   revenue_CLEANED                           91430 non-null  int64  
 6   arpu_segment_CLEANED                      91430 non-null  int64  
 7   frequence_CLEANED                         91430 non-null  int64  
 8   data_volume_CLEANED                       91430 non-null  int64  
 9   on_net_CLEANED                      

Trying the dataset found from IBM:
https://www.kaggle.com/datasets/blastchar/telco-customer-churn/data

In [ ]:
# !python -m pip install --upgrade pip
# !pip install scikit-learn==1.8.0
# while True:
#   pass

In [10]:
import sklearn
print(sklearn.__version__)

1.6.1


In [ ]:
#https://drive.google.com/file/d/1Bv0q9pV0sOLHQlS7x7OaACvdEHPLRcHM/view?usp=drive_link

1. Anomaly detection

In [11]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

In [ ]:
!wget https://media.geeksforgeeks.org/wp-content/uploads/20240402160319/creditcard.csv
#Just some data given from the tutorial I used*

In [12]:
tele_forest = fully_cleaned_teleset[[column for column in fully_cleaned_teleset.keys() if str(fully_cleaned_teleset[column].dtype) != 'object']]

'''When I used the I.F on the Expresso dataset (you can also find it here: https://www.kaggle.com/datasets/hamzaghanmi/expresso-churn-prediction-challenge/data)
Some columns were almost full of anomalies, but previously I used the RF as a test and found the accuracy near 100% ... Is that contradictory to overfitting factors?'''
# NAs = [column for column in credit_data.keys() if sum(list(credit_data[column].isna()))]
# for column in expressed_data.keys():

#   print(column," = ",sum(list(expressed_data[column].isna())))
# credit_data = credit_data.dropna(subset=NAs)


'When I used IF on the Expresso dataset (you can also find it here: https://www.kaggle.com/datasets/hamzaghanmi/expresso-churn-prediction-challenge/data)\nSome columns were almost full of anomalies, but previously I used the RF as a test and found the accuracy near 100% ... Is that contradictory to overfitting factors?'

In [30]:
scaler = StandardScaler().fit_transform(tele_forest.loc[:,tele_forest.columns!='churn']) #NOTE: that is an evidence that teh I.F works only with num_data
scaled_data = scaler[0:len(tele_forest.values)]
df = pd.DataFrame(data=scaled_data)
inlined_tele_forest = tele_forest.drop(index=test_set.index)

X = tele_forest.drop(columns=['churn'])
y = tele_forest['churn']

In [14]:
outlier_fraction = len(tele_forest[tele_forest['churn']==1])/float(len(tele_forest[tele_forest['churn']==0]))
model =  IsolationForest(n_estimators=1000, contamination=outlier_fraction, random_state=50)
model.fit(df)

IsolationForest(contamination=0.25382262996941896, n_estimators=1000,
                random_state=50)

In [15]:
scores_prediction = model.decision_function(df)
y_pred = model.predict(df)
print(y_pred.shape, tele_forest.shape)
y_pred[y_pred == 1] = 0
y_pred[y_pred == -1] = 1
print("Accuracy in finding anomaly:",accuracy_score(y,y_pred))

outlier_s = {'outliers':[(Y,int(tele_forest['churn'].iloc[Y])) for Y in range(len(tele_forest.index)) if y_pred[Y]]}
print(len([ i for i in outlier_s['outliers'] if i[1] ]), len([ i for i in outlier_s['outliers'] if not i[1] ]))

(91430,) (91430, 21)
Accuracy in finding anomaly: 0.5515257574100405
356 22851


In [37]:
def outlier_summary(data, cols):
    rows = []
    for col in cols:
        s = data[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        # n_out = ((s < lo) | (s > hi)).sum()
        mask = (s < lo) | (s > hi)
        n_out = mask.sum()

        # 2. Get the index values only for this column
        # This returns a simple LIST of integers
        ref_indices = s.index[mask].tolist()

        rows.append({
            "feature":   col,
            "Q1":        round(q1, 2),
            "Q3":        round(q3, 2),
            "IQR":       round(iqr, 2),
            "lower_fence": round(lo, 2),
            "upper_fence": round(hi, 2),
            "n_outliers":  n_out,
            "references": ref_indices#tele_forest.apply(lambda x: x.index[(x < lo) | (x > hi)].tolist()),
        })
    return pd.DataFrame(rows).sort_values("n_outliers", ascending=False)

df_test = outlier_summary(tele_forest,tele_forest.columns)
# [ref for ref in df_test['references'].to_list()]# - 18509 #Betoo3 Elli churN



user_id = list({j for i in df_test['references'] for j in i})#set(j for j in [i for i in list(df_test['references'])])
# user_id.sort
user_id_len = len(user_id)
tele_id_lst = list(tele_forest.index)
# print(user_id_len)
tele_No = 0
# for Y in range(user_id_len):
#   # print(id)
#   for id in range(tele_No,len(tele_id_lst)):
#     if user_id[Y] == tele_id_lst[id]:
#       user_id[Y] = tele_No
#       # print(user_id[Y])
#       break
#     tele_No += 1
    # print(user_id[Y])
# user_id = [list(tele_forest.index).index(Y) for Y in user_id]

tuple(user_id)

('...',
 ...)

2. Class Prediction

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings

In [17]:
# warnings.filterwarnings('ignore')
expressed_data = pd.read_csv('https://drive.google.com/uc?id=1Bv0q9pV0sOLHQlS7x7OaACvdEHPLRcHM')

rf_classifier = RandomForestClassifier(n_estimators=1000, random_state=50)


In [18]:
fully_cleaned_teleset[fully_cleaned_teleset.index == "0a7a8f74861cc45a415523e1f93904cb329e601b"]

,region_CLEANED,tenure_symbol,tenure_value,montant_CLEANED,frequence_rech_CLEANED,revenue_CLEANED,arpu_segment_CLEANED,frequence_CLEANED,data_volume_CLEANED,on_net_CLEANED,...,TOP_PACK_id,freq_top_pack_CLEANED,region_tower_count_CLEANED,region_avg_range_CLEANED,region_avg_samples_CLEANED,cvrg_index_CLEANED,region_network_quality_score_CLEANED,department_network_quality_score_CLEANED,arr_network_quality_score_CLEANED,churn
user_id,,,,,,,,,,,,,,,,,,,,,
0a7a8f74861cc45a415523e1f93904cb329e601b,DAKAR,K,25,10500,34,10364,3455,37,10870,7,...,15,26,1771,1347.316206,6.294184,2386097,533.188255,134.914073,55.1061,0


In [21]:
!nvidia-smi #Just a place_holder if you want start using the T4 GPU.

/bin/bash: line 1: nvidia-smi: command not found


In [40]:
test_set = fully_cleaned_teleset.iloc[ list(map(
                                       lambda x: x[0],outlier_s['outliers']))]
train_set = fully_cleaned_teleset.drop(index=test_set.index)

# ayman_test_set = fully_cleaned_teleset.loc[user_id]
# ayman_train_set = fully_cleaned_teleset.drop(index = ayman_test_set.index)
# print(ayman_test_set.shape , ayman_train_set.shape); exit(0)
X_train = train_set[[column for column in train_set.keys()[:-1] if str(train_set[column].dtype) != 'object']]
y_train = train_set[train_set.columns[-1]]


X_test = test_set[[column for column in test_set.keys()[:-1] if str(test_set[column].dtype) != 'object']]
y_test = test_set[test_set.columns[-1]]

# X['Sex'] = X['Sex'].map({'female': 0, 'male': 1}) NOTE: no mapping required
# X['Age'] = X['Age'].fillna(X['Age'].median()) NOTE: no NAs to be filled

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)#user_id)/

rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:\n", classification_rep)

# sample = X_test.iloc[0:1]
# prediction = rf_classifier.predict(sample)


Accuracy: 0.86

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.91      0.91     36481
           1       0.66      0.67      0.66      9234

    accuracy                           0.86     45715
   macro avg       0.79      0.79      0.79     45715
weighted avg       0.86      0.86      0.86     45715



Accuracy: 0.82 (0.2 split)

Classification Report:

               precision    recall  f1-score   support

           0       0.88      0.87      0.88     10018
           1       0.65      0.68      0.66      3627

    accuracy                           0.82     13645
    macro avg       0.77      0.77      0.77     13645
    weighted avg       0.82      0.82      0.82     13645


Accuracy: 0.86 (ayman_outs)

Classification Report:

               precision    recall  f1-score   support

           0       0.92      0.91      0.91     62606
           1       0.65      0.69      0.67     15812

    accuracy                           0.86     78418
    macro avg       0.79      0.80      0.79     78418
    weighted avg    0.87      0.86      0.86     78418


Accuracy: 0.86 (I.F outs)

Classification Report:

               precision    recall  f1-score   support

           0       0.91      0.92      0.91     18573
           1       0.66      0.66      0.66      4634

    accuracy                           0.86     23207
    macro avg       0.79      0.79      0.79     23207
    weighted avg    0.86      0.86      0.86     23207

In [24]:
print(len(test_set),len(train_set))
X_test[:100]
print(list(y_test[0:100]).index(1))

23207 68223
2


In [28]:
# print(X_test[:10])
# print(y_test[:10])
in_sample = X_test[:100]
prediction = rf_classifier.predict(in_sample)
for i in range (len(in_sample)):
  sample_dict = in_sample.iloc[i].to_dict()
  print(f"\nSample Passenger: {sample_dict}")
  print(prediction)
  print(f"Predicted Survival for : {'Churn' if prediction[i] == 1 else 'No churn'}")


Sample Passenger: {'tenure_value': 25.0, 'montant_CLEANED': 8650.0, 'frequence_rech_CLEANED': 37.0, 'revenue_CLEANED': 8650.0, 'arpu_segment_CLEANED': 2883.0, 'frequence_CLEANED': 38.0, 'data_volume_CLEANED': 2593.0, 'on_net_CLEANED': 3.0, 'orange_CLEANED': 39.0, 'tigo_CLEANED': 1.0, 'regularity': 59.0, 'TOP_PACK_id': 14.0, 'freq_top_pack_CLEANED': 20.0, 'region_tower_count_CLEANED': 1771.0, 'region_avg_range_CLEANED': 1347.3162055335968, 'region_avg_samples_CLEANED': 6.294184076792773, 'cvrg_index_CLEANED': 2386097.0, 'region_network_quality_score_CLEANED': 533.1882552230378, 'department_network_quality_score_CLEANED': 134.91407313539614, 'arr_network_quality_score_CLEANED': 55.10609965302018}
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Predicted Survival for : No churn

Sample Passenger: {'tenure_value': 25.0, 'montant_CLEANE

In [ ]:
almost_cleaned_teleset.index[0]

'22322777e909ae0aa1c69e7b1fce40f333187fa6'

In [ ]:
train_set = almost_cleaned_teleset.iloc[ list(map(
                                       lambda x: x[0],outlier_s['outliers']))]
train_set.index
# almost_cleaned_teleset.index

Index(['22322777e909ae0aa1c69e7b1fce40f333187fa6',
       '453d2b534a53e30171b3c0c72dc7d56415619d8a',
       '9e64b12fd171b51378111bbb1f25242e79c7be99',
       '369f886e40ef9ec4de23784888d412e5f336cca1',
       '3524a53cd138dfe31a2f48c2cc21ed4129df1d10',
       'c654eb8aaff74cb2f13fbcb191b4efe5590f383b',
       'de884b175a5993a4e57d27cf731260439ece7648',
       '42089fc2f8bf952fed51e256031404e2ac8204eb',
       'a2bef58fac4898de52c9371f152441c4bbe79bb5',
       '39f66cd56ee182d98ce301c446b33683a43ef027',
       ...
       '4d8009065c81f36556ce45009b26aaf00b20ae69',
       'cbf23cff8ed8ca535461e2fd830839812d8b2895',
       '7d20173c4e65b86a92df2dbf3a6fca510ea87850',
       '464fa7bb08c16b9c81e7fd6e25f6191a52f37694',
       '60749c2d6b699021054533734e3c9e38e8ef99b1',
       '6d0364a152b97c501419d9f024188c9a235d1d16',
       '45d8d1888630f6c019b90ea94273ec9d90c56338',
       '1056a3ea4257e519985070895f562feea5a4cac6',
       '71ce8a04aab2bb06e7f03a9af9d648e7bbba54f6',
       '32e30b9357c9

# Develop inference scripts to predict churn probability for dirty_Data_100k.

In [ ]:
#https://drive.google.com/file/d/1WpQ3o4vWdUb3QN6nuw4WX7pMgKNWV49t/view

In [ ]:
warnings.filterwarnings('ignore')
full_data = pd.read_csv('https://drive.google.com/uc?id=1WpQ3o4vWdUb3QN6nuw4WX7pMgKNWV49t')

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)


In [ ]:
full_data.info()

NAs = [column for column in full_data.keys() if sum(list(full_data[column].isna()))]
# for column in expressed_data.keys():

#   print(column," = ",sum(list(expressed_data[column].isna())))
full_data = full_data.dropna(subset=NAs)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 32 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   user_id                           100000 non-null  object 
 1   region                            60844 non-null   object 
 2   tenure                            100000 non-null  object 
 3   montant                           65059 non-null   float64
 4   frequence_rech                    65059 non-null   float64
 5   revenue                           66484 non-null   float64
 6   arpu_segment                      66484 non-null   float64
 7   frequence                         66484 non-null   float64
 8   data_volume                       50813 non-null   float64
 9   on_net                            63663 non-null   float64
 10  orange                            58621 non-null   float64
 11  tigo                              40369 non-null   fl

In [ ]:
X = full_data[[column for column in full_data.keys()[:-1] if str(full_data[column].dtype) != 'object']]
y = full_data[full_data.columns[-1]]

# X['Sex'] = X['Sex'].map({'female': 0, 'male': 1}) NOTE: no mapping required
# X['Age'] = X['Age'].fillna(X['Age'].median()) NOTE: no NAs to be filled

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:\n", classification_rep)

sample = X_test.iloc[0:1]
prediction = rf_classifier.predict(sample)

sample_dict = sample.iloc[0].to_dict()
print(f"\nSample Passenger: {sample_dict}")
print(f"Predicted Survival: {'Churn' if prediction[0] == 1 else 'No churn'}")

Accuracy: 0.99

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99        98
           1       0.00      0.00      0.00         1

    accuracy                           0.99        99
   macro avg       0.49      0.50      0.50        99
weighted avg       0.98      0.99      0.98        99


Sample Passenger: {'montant': 8550.0, 'frequence_rech': 16.0, 'revenue': 8549.0, 'arpu_segment': 2850.0, 'frequence': 20.0, 'data_volume': 6349.0, 'on_net': 3.0, 'orange': 93.0, 'tigo': 2.0, 'zone1': 1.0, 'zone2': 0.0, 'regularity': 52.0, 'freq_top_pack': 5.0, 'region_tower_count': 1771.0, 'region_avg_range': 1347.3162055335968, 'region_avg_samples': 6.294184076792773, 'region_avg_signal': 0.0, 'region_coverage_index': 2386097.0, 'region_signal_strength_index': 0.0, 'region_network_quality_score': 533.1882552230378, 'department_signal_strength_index': 0.0, 'department_network_quality_score': 134.91407313539614, 'department_c